In [21]:
# Cell 0: synchronize the repository environment before running the workbench.
from pathlib import Path
import subprocess

SETUP_ROOT = Path.cwd()
while not (SETUP_ROOT / "pyproject.toml").exists() and SETUP_ROOT != SETUP_ROOT.parent:
    SETUP_ROOT = SETUP_ROOT.parent

if not (SETUP_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Could not find the repository root containing pyproject.toml.")

print("Running: uv sync --all-extras --dev --all-packages")
setup_result = subprocess.run(
    ["uv", "sync", "--all-extras", "--dev", "--all-packages"],
    cwd=SETUP_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.returncode != 0:
    print(setup_result.stderr)
    raise RuntimeError(
        "uv sync failed. Install uv or restart the notebook with the repository environment selected."
    )
print("Environment synchronized successfully.")
print("If packages changed, restart the notebook kernel before continuing.")

Running: uv sync --all-extras --dev --all-packages

Environment synchronized successfully.
If packages changed, restart the notebook kernel before continuing.


# Manufacturing Stress Forecasting Workbench

This notebook is the main interactive entry point for the manufacturing-stress experiment. It creates or refreshes the FRED data cache, runs the deterministic baselines, and optionally runs the token-limited LLMP backtest.

All predictors use the same backtest specification and shared Brier-score implementation. The LLMP is opt-in and cached so repeated analysis does not make new model calls.

In [ ]:
# Main controls: change these values, then run the notebook from top to bottom.
REFRESH_FRED_DATA = False
BACKTEST_STRIDE = 3
RUN_LLMP_BACKTEST = False
FORCE_REFRESH_LLMP_CACHE = False
RUN_CURRENT_FORECAST = False

# BACKTEST_STRIDE = 1 evaluates every month; 3 evaluates every third month.
# False: one smoke run for historical frequency, logistic regression, and XGBoost.
# True: the same smoke run also includes the cached, token-limited LLMP.
# RUN_CURRENT_FORECAST controls the separate latest-data forecast section.
print({
    "refresh_fred_data": REFRESH_FRED_DATA,
    "backtest_stride": BACKTEST_STRIDE,
    "run_llmp_backtest": RUN_LLMP_BACKTEST,
    "force_refresh_llmp_cache": FORCE_REFRESH_LLMP_CACHE,
    "run_current_forecast": RUN_CURRENT_FORECAST,
})

{'refresh_fred_data': False, 'backtest_stride': 3, 'run_llmp_backtest': True, 'force_refresh_llmp_cache': True, 'run_current_forecast': False}


## Imports and project paths

Run this notebook with the repository environment selected as the Jupyter kernel. The `.env` file is loaded for the FRED key and, when needed, the LLM proxy credentials.

In [13]:
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
from aieng.forecasting.evaluation import BacktestSpec, backtest
from aieng.forecasting.evaluation.artifacts import load_backtest_result, save_backtest_result
from aieng.forecasting.methods import HistoricalFrequencyPredictor
from manufacturing_stress_forecasting.analyst_agent import build_manufacturing_stress_agent_predictor
from manufacturing_stress_forecasting.data import IPMAN_SERIES_ID, build_manufacturing_stress_service
from manufacturing_stress_forecasting.predictors import (
    ManufacturingStressLogisticPredictor,
    ManufacturingStressXGBoostPredictor,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().parents[1]

load_dotenv(REPO_ROOT / ".env", override=False)
SPEC_PATH = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
STORE_DIR = REPO_ROOT / "data" / "predictions"
LLMP_SPEC_ID = "manufacturing_stress_smoke_llmp_v1"
print(f"Repository: {REPO_ROOT}")

Repository: c:\Users\Mina Attari\BMO-C2-agentic-forecasting


## Create or refresh the data

The service loads `IPMAN`, `DFF`, `DGS10`, and `DGS2` from the FRED cache. Set `REFRESH_FRED_DATA = True` in the first cell when the cache should be updated from FRED.

In [15]:
service = build_manufacturing_stress_service(
    cache_dir=REPO_ROOT / "data" / "fred",
    refresh=REFRESH_FRED_DATA,
)
print(service.summary().to_string(index=False))

                        series_id                                                                                       description                                   source                                units frequency  n_obs      start        end
               fed_funds_rate_pct                                                            Month-end effective federal funds rate              FRED (DFF), derived monthly                              Percent        MS    867 1954-07-01 2026-09-01
              ipman_change_1m_pct                                                       Trailing 1-month percentage change in IPMAN                  Derived from FRED IPMAN                              Percent        MS    654 1972-02-01 2026-07-01
              ipman_change_3m_pct                                                       Trailing 3-month percentage change in IPMAN                  Derived from FRED IPMAN                              Percent        MS    652 1972-04-01 2026-07-01
    

## Load the common backtest specification

Every predictor below receives this same target, horizon, forecast-origin schedule, warmup, and cutoff-aware data service.

In [16]:
with SPEC_PATH.open() as file:
    spec = BacktestSpec.model_validate(yaml.safe_load(file))
spec = spec.model_copy(update={"stride": BACKTEST_STRIDE})

print(spec.task.description)
print(f"Origins: {spec.start.date()} to {spec.end.date()}, stride={spec.stride}, warmup={spec.warmup}")
print(f"Target: {spec.task.target_series_id}; horizon={spec.task.horizons[0]} {spec.task.frequency}")

Probability that U.S. manufacturing will be under stress three months ahead. A resolved month is stressed when IPMAN has declined by at least 2 percent over its preceding three months.
Origins: 2018-01-01 to 2024-12-01, stride=3, warmup=60
Target: manufacturing_stress; horizon=3 MS


## Run one smoke test

This is the single execution path. It always runs the historical-frequency, logistic-regression, and XGBoost predictors. Set `RUN_LLMP_BACKTEST = True` in the first cell to include the token-limited LLMP in the same run. When LLMP mode is enabled, completed results are loaded from cache unless `FORCE_REFRESH_LLMP_CACHE = True`.

In [17]:
def run_cached_backtest(predictor, *, force_refresh=False):
    predictor_id = predictor.predictor_id
    artifact_path = STORE_DIR / LLMP_SPEC_ID / f"{predictor_id}.yaml"
    if not force_refresh:
        cached = load_backtest_result(LLMP_SPEC_ID, predictor_id, store_dir=STORE_DIR)
        if cached is not None:
            print(f"{predictor_id}: loaded {artifact_path}")
            return cached

    result = backtest(
        predictor=predictor,
        spec=spec,
        data_service=service,
        max_retries=1,
        retry_delay=1.0,
    )
    path = save_backtest_result(result, spec_id=LLMP_SPEC_ID, store_dir=STORE_DIR)
    print(f"{predictor_id}: saved {path}")
    return result


predictors = [
    HistoricalFrequencyPredictor(),
    ManufacturingStressLogisticPredictor(),
    ManufacturingStressXGBoostPredictor(),
]
if RUN_LLMP_BACKTEST:
    predictors.append(build_manufacturing_stress_agent_predictor())

results = {}
for predictor in predictors:
    if RUN_LLMP_BACKTEST:
        result = run_cached_backtest(predictor, force_refresh=FORCE_REFRESH_LLMP_CACHE)
    else:
        result = backtest(predictor=predictor, spec=spec, data_service=service)
    results[predictor.predictor_id] = result
    print(
        f"{result.predictor_id}: {result.mean_score:.4f} mean {result.metric}; "
        f"scored={len(result.scores)} skipped={result.skipped_origins}"
    )

historical_frequency: saved c:\Users\Mina Attari\BMO-C2-agentic-forecasting\data\predictions\manufacturing_stress_smoke_llmp_v1\historical_frequency.yaml
historical_frequency: 0.0356 mean brier; scored=28 skipped=0
manufacturing_stress_logistic_ipman_rates: saved c:\Users\Mina Attari\BMO-C2-agentic-forecasting\data\predictions\manufacturing_stress_smoke_llmp_v1\manufacturing_stress_logistic_ipman_rates.yaml
manufacturing_stress_logistic_ipman_rates: 0.0471 mean brier; scored=28 skipped=0


Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


manufacturing_stress_xgboost_ipman_rates: saved c:\Users\Mina Attari\BMO-C2-agentic-forecasting\data\predictions\manufacturing_stress_smoke_llmp_v1\manufacturing_stress_xgboost_ipman_rates.yaml
manufacturing_stress_xgboost_ipman_rates: 0.0413 mean brier; scored=28 skipped=0


Node execution failed with exception
Traceback (most recent call last):
  File "c:\Users\Mina Attari\BMO-C2-agentic-forecasting\.venv\Lib\site-packages\google\adk\workflow\_node_runner.py", line 135, in run
    await self._execute_node(ctx, node_input)
  File "c:\Users\Mina Attari\BMO-C2-agentic-forecasting\.venv\Lib\site-packages\google\adk\workflow\_node_runner.py", line 255, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "c:\Users\Mina Attari\BMO-C2-agentic-forecasting\.venv\Lib\site-packages\google\adk\workflow\_node_runner.py", line 269, in _run_node_loop
    async for event in agen:
  File "c:\Users\Mina Attari\BMO-C2-agentic-forecasting\.venv\Lib\site-packages\google\adk\workflow\_base_node.py", line 217, in run
    async for item in agen:
  File "c:\Users\Mina Attari\BMO-C2-agentic-forecasting\.venv\Lib\site-packages\google\adk\agents\llm_agent.py", line 559, in _run_impl
    async for event in agen:
  File "c:\Users\Mina Attari\BMO-C2-agentic-forecastin

ValueError: No predictions were scored. All 28 candidate origins were skipped. Check that the target series covers the evaluation window and that warmup (60) is not too large.

## Compare results

Use only results with matching scored-origin counts for a fair comparison. Lower Brier score is better.

In [18]:
comparison = pd.DataFrame(
    [
        {
            "predictor": result.predictor_id,
            "metric": result.metric,
            "mean_brier": result.mean_score,
            "scored": len(result.scores),
            "skipped": result.skipped_origins,
        }
        for result in results.values()
    ]
)
comparison.sort_values("mean_brier") if not comparison.empty else comparison

,predictor,metric,mean_brier,scored,skipped
0,historical_frequency,brier,0.035598,28,0
2,manufacturing_stress_xgboost_ipman_rates,brier,0.041290,28,0
1,manufacturing_stress_logistic_ipman_rates,brier,0.047142,28,0


## Optional current forecast

Set `RUN_CURRENT_FORECAST = True` in the first cell to run the selected predictors against the latest released IPMAN data. This is a current forecast, not a historical backtest, so it does not produce a Brier score. `RUN_LLMP_BACKTEST` controls whether the LLMP is included here too.

In [19]:
if RUN_CURRENT_FORECAST:
    with SPEC_PATH.open() as file:
        current_task = BacktestSpec.model_validate(yaml.safe_load(file)).task

    full_ipman = service.get_series(
        IPMAN_SERIES_ID,
        as_of=pd.Timestamp("2100-01-01").to_pydatetime(),
    )
    current_as_of = pd.Timestamp(full_ipman["released_at"].max())
    current_context = service.context(as_of=current_as_of.to_pydatetime())

    current_predictors = [
        HistoricalFrequencyPredictor(),
        ManufacturingStressLogisticPredictor(),
        ManufacturingStressXGBoostPredictor(),
    ]
    if RUN_LLMP_BACKTEST:
        current_predictors.append(build_manufacturing_stress_agent_predictor())

    print(f"Forecast origin: {current_as_of.date()}")
    print(
        "Latest visible IPMAN reference month: "
        f"{pd.Timestamp(current_context.get_series(IPMAN_SERIES_ID)['timestamp'].max()).date()}"
    )
    for predictor in current_predictors:
        prediction = predictor.predict(current_task, current_context)[0]
        print({
            "predictor": prediction.predictor_id,
            "forecast_date": str(pd.Timestamp(prediction.forecast_date).date()),
            "stress_probability": prediction.payload.probability,
            "metadata": prediction.metadata,
        })
else:
    print("Current forecast is disabled. Set RUN_CURRENT_FORECAST = True to enable it.")

Current forecast is disabled. Set RUN_CURRENT_FORECAST = True to enable it.
